<a href="https://colab.research.google.com/github/uday5t61/ai-agents/blob/main/Travel_Planner_Agent_with_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --no-cache-dir openai-agents==0.2.2
!pip install python-dotenv langchain-openai==0.2.1

!pip install --upgrade pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.1 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.1 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
langchain 1.3.1 requires langchai

In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from google.colab import userdata


openai_api_key = userdata.get('OPENAI_API_KEY')

load_dotenv()
os.environ['OPENAI_API_KEY'] = openai_api_key
openai_client = OpenAI(
  api_key=openai_api_key
)


In [4]:
from IPython.display import display, Markdown
def print_markdown(text):
    display(Markdown(text))

In [6]:
from agents import Agent,Runner,SQLiteSession

from agents import set_tracing_export_api_key

# Explicitly pass the key to the tracing module
set_tracing_export_api_key(openai_api_key)

market_researcher_instructions = """
Context:
You are a market research assistant helping analyze companies, industries, and competitors.

Instructions:
When given a question, provide a short factual answer based on your knowledge.

Output:
Start with a verdict prefix: either "✅ FACT:" or "❌ UNKNOWN:"
Follow with a concise one-sentence explanation.
"""

market_researcher_agent = Agent(name="Marker Researcher",
                                instructions=market_researcher_instructions,
                                model="gpt-4.1-mini")

In [7]:
#AI Agent with no memeory

q1= "What is the market share of Tesla in the US EV market?"

print_markdown(f"You: '{q1}'")

resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1)

print_markdown(f"🤖 Agent:\n{resp1.final_output}")

You: 'What is the market share of Tesla in the US EV market?'

🤖 Agent:
✅ FACT: Tesla holds approximately 65-70% of the US electric vehicle market share as of early 2024.

In [8]:
q1= "How does that compare to last year?"

print_markdown(f"You: '{q1}'")

resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1)

print_markdown(f"🤖 Agent:\n{resp1.final_output}")

You: 'How does that compare to last year?'

🤖 Agent:
❌ UNKNOWN: You have not provided any specific data or context to compare this year to last year.

## Adding Memory to the agent

In [9]:
# Create a session instance
# SQLite-based implementation of session storage.
# This implementation stores conversation history in a SQLite database.

session = SQLiteSession("conversation")

In [10]:
q1= "What is the market share of Tesla in the US EV market?"

print_markdown(f"You: '{q1}'")

resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1,session=session)

print_markdown(f"🤖 Agent:\n{resp1.final_output}")

You: 'What is the market share of Tesla in the US EV market?'

🤖 Agent:
✅ FACT: Tesla holds around 60-70% of the US electric vehicle (EV) market share as of early 2024.

In [11]:
q2= "How does that compare to last year?"

print_markdown(f"You: '{q2}'")

resp2 = await Runner.run(starting_agent = market_researcher_agent, input = q2,session=session)

print_markdown(f"🤖 Agent:\n{resp2.final_output}")

You: 'How does that compare to last year?'

🤖 Agent:
✅ FACT: Tesla's US EV market share has slightly declined from about 70-75% last year to around 60-70% in early 2024 due to increased competition.

# Travel Planner Agent

In [21]:
travel_planner_instructions = """
Context: You are an expert travel planner assistant.

Instructions: When asked for a recommendation, you must suggest exactly one sunny and warm weekend getaway destination. Do not provide multiple options.

"""

In [22]:
travel_session = SQLiteSession("travel_planner")

travel_planner_agent = Agent(name="Travel Planner",
                                instructions=travel_planner_instructions,
                                model="gpt-5-mini")


In [23]:
q1= "Please suggest one weekend destination within 2 hours by road trip from Hyderabad, India"

print_markdown(f"You: '{q1}'")

resp1 = await Runner.run(starting_agent = travel_planner_agent, input = q1,session=travel_session)

print_markdown(f"🤖 Agent:\n{resp1.final_output}")

You: 'Please suggest one weekend destination within 2 hours by road trip from Hyderabad, India'

🤖 Agent:
Suggestion: Ananthagiri Hills (Vikarabad)

Why go: About 80–95 km from Hyderabad (roughly 1.5–2 hours by road), Ananthagiri Hills is a sunny, green weekend escape with gentle hillwalks, coffee plantations, a scenic viewpoint for sunrise, and the ancient Anantha Padmanabha Temple — perfect for a short, warm outdoors weekend.

Quick 1-night itinerary
- Day 1 (morning): Leave Hyderabad early (6–7 AM). Reach Ananthagiri, check into a homestay/resort. Breakfast in Vikarabad on the way.  
- Day 1 (late morning/afternoon): Walk through coffee estates, short trek to the viewpoint and nearby forests. Picnic/lunch at the resort or local eateries.  
- Day 1 (evening): Sunset/viewpoint, explore the temple, relax at the property.  
- Day 2 (morning): Early nature walk or short trek, breakfast, return to Hyderabad by late morning/early afternoon.

Highlights
- Coffee plantations and rustic village scenery  
- Short treks and viewpoints (good for sunrise/sunset)  
- Anantha Padmanabha Temple and local forested pockets  
- Quiet, less crowded than larger hill stations

Practical tips
- Drive time: ~1.5–2 hours depending on traffic; leave early to avoid city traffic.  
- Pack: sunscreen, hat, comfortable shoes, water, light layers for mornings.  
- Stay: small resorts or homestays in/around Ananthagiri or Vikarabad.  
- Best time: year-round; for warm sunny weekends aim for post-winter or pre-monsoon months.

If you want, I can draft turn-by-turn directions, suggest a specific homestay/resort, or book a simple itinerary for this trip.

In [24]:
q2= "How much does it cost to go by car"

print_markdown(f"You: '{q2}'")

resp2 = await Runner.run(starting_agent = travel_planner_agent, input = q2,session=travel_session)

print_markdown(f"🤖 Agent:\n{resp2.final_output}")

You: 'How much does it cost to go by car'

🤖 Agent:
Do you mean driving your own car, renting a self‑drive, or hiring a cab? I’ll give quick estimates for all three using Hyderabad → Ananthagiri Hills (round trip ≈ 160–200 km; use 180 km for calculations).

Assumptions (for clarity)
- Round trip distance: 180 km  
- Petrol price: ~₹105–125/litre (use ₹115 as midpoint)  
- Diesel price: ~₹90–105/litre (use ₹98 as midpoint)  
- Typical mileage: petrol car ~14 km/l, diesel car ~18 km/l

1) Your own car (fuel + tolls + parking)
- Petrol car: fuel ≈ (180/14) × ₹115 ≈ ₹1,480 (range ≈ ₹1,350–1,610)  
- Diesel car: fuel ≈ (180/18) × ₹98 ≈ ₹980 (range ≈ ₹900–1,150)  
- Tolls & parking: ₹0–₹150  
Estimated total: Petrol ≈ ₹1,350–1,750; Diesel ≈ ₹900–1,300

2) Self‑drive rental (1 day)
- Small hatchback: rental ₹1,200–2,000/day + fuel (~₹1,400) + tolls ₹0–150  
Estimated total: ₹2,600–3,600
- Sedan/UV: add ~₹500–1,000 to rental

3) Outstation cab (round trip, driver included)
- Sedan (Swift Dzire / equivalent): ₹3,000–4,500 for a day trip  
- Innova / bigger SUV: ₹4,500–6,500  
(These fares usually include driver charges and limited kms; confirm extras like night halt or extra km.)

Other minor costs: driver allowance ₹300–600 (if applicable), snacks/parking extras.

If you tell me which option you prefer (own car, self‑drive, or cab) and your car type or luggage/comfort needs, I can give a tighter estimate and/or find specific rental/cab prices.